In [ ]:
JOBS_GPU0 = ['C:icarl:s0', 'C:icarl:s2', 'C:icarl_full:s1', 'C:icarl_cbp:s0', 'C:icarl_cbp:s2', 'C:icarl_redo:s1', 'C:icarl_snp:s0', 'C:icarl_snp:s2', 'C:icarl_upgd:s1', 'C:gem:s0', 'C:gem:s2', 'C:wa:s1', 'C:podnet:s0', 'C:podnet:s2', 'C:icarl_cflat:s1']
JOBS_GPU1 = ['C:icarl:s1', 'C:icarl_full:s0', 'C:icarl_full:s2', 'C:icarl_cbp:s1', 'C:icarl_redo:s0', 'C:icarl_redo:s2', 'C:icarl_snp:s1', 'C:icarl_upgd:s0', 'C:icarl_upgd:s2', 'C:gem:s1', 'C:wa:s0', 'C:wa:s2', 'C:podnet:s1', 'C:icarl_cflat:s0', 'C:icarl_cflat:s2']
JOBS_CPU = []
RUN_TAG = 'clip_study'
PRESET = 'CLIP_C'
REPO_COMMIT = 'c42995dd101466d66ca9809a00a986841b7d6322'
SOURCE_DIGEST = None
GRAD_CLIP = None  # Registered runs stay unclipped; selected tau is authorized only by the clip-study C gate.
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
ALLOW_OVER_BUDGET = True
DISK_CAP_BYTES = 5_000_000_000

PHASE = 'C'
DISK_CAP_BYTES = 20_000_000_000


# Gradient clipping study — Phase C
Attach CIFAR-100 with cifar-100-python/{train,test,meta}, plus the four original historical archives: M1 seeds1/2, C3, C2 and Gram seed0. Reattach all four original archives on EVERY resumed session; imported _sources are deliberately excluded from output archives. The pinned source must include experiments/clip_study.py; regenerate from the published study commit before running. Also attach the A/B output archive containing all A/B reports, grad_clip_selection.json and PREREGISTRATION_CHANGES.md. The selection is recomputed and checked before any Phase C job starts. Both GPUs process the authorized 30 cases. No manual tau override is accepted. The 11.5-hour deadline pauses safely; rerunning continues cached/native checkpoints. This notebook archives only clip_study results, checkpoints and logs; it does not copy other historical runs into the output archive.

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/vna4_clip_C")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if len(branch) == 40 and all(c in "0123456789abcdef" for c in branch):
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", branch], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", branch], env=env, check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
DATA_ROOT = Path('/kaggle/input')
MNIST_ROOT = next((p.parent for p in Path('/kaggle/input').rglob('mnist_') if p.is_file()),None)
if JOBS_GPU0 or JOBS_GPU1:
    import torch
    if torch.cuda.device_count() != 2:
        raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
    required_cifar_files = {"train", "test", "meta"}
    cifar_dirs = []
    for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
        path = Path(directory)
        if (path.name == "cifar-100-python" and
                required_cifar_files.issubset(filenames)):
            cifar_dirs.append(path.parent)
    cifar_dirs = sorted(set(cifar_dirs), key=str)
    if not cifar_dirs:
        visible = []
        for directory, dirnames, filenames in os.walk(
                "/kaggle/input", followlinks=True):
            visible.append({
                "directory": directory,
                "directories": sorted(dirnames)[:10],
                "files": sorted(filenames)[:10],
            })
            if len(visible) == 20:
                break
        raise FileNotFoundError(
            "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
            f"Visible input inventory: {visible}")
    DATA_ROOT = cifar_dirs[0]
    print("CIFAR-100 root:", DATA_ROOT)
    
if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]



subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quadprog"], check=True)


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), STAGE_B/'clip_study', 'clip_study', exclude_parts=('_sources',))
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs/clip_study",), exclude_parts=("data","_sources"))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
from experiments.clip_study import load_selection
from experiments.vna_resume import launch_logged
study_root = STAGE_B/'clip_study'
study_root.mkdir(parents=True,exist_ok=True)
selection = study_root/'grad_clip_selection.json'
try:
    if PHASE == 'C':
        chosen=load_selection(selection)
        if chosen['code_commit'] != REPO_COMMIT:
            raise ValueError('Phase C requires the exact A/B source commit')
        if not (study_root/'PREREGISTRATION_CHANGES.md').exists():
            raise FileNotFoundError('Attach A/B output with selection evidence and its written preregistration')
        print('Phase C authorized threshold:',chosen['tau'],'G_max:',chosen['G_max'],flush=True)
        import re
        producers=set()
        for path in (study_root/'_references').glob('*.json'):
            descriptor=json.loads(path.read_text());identity=descriptor.get('identity',{})
            if descriptor.get('epochs') or identity.get('arm') in ('wa','podnet','icarl_cflat'):continue
            if identity.get('arm')=='gem' and identity.get('seed')!=0:continue
            commit=identity.get('code_commit')
            if commit and commit!=REPO_COMMIT:producers.add(commit)
        for commit in sorted(producers):
            if not re.fullmatch('[0-9a-f]{40}',commit):raise ValueError('Invalid historical producer commit')
            target=REPO.with_name('clip_reference_'+commit[:12])
            if not target.exists():private_clone(MAIN_URL,target,commit)
            head=subprocess.check_output(['git','-C',str(target),'rev-parse','HEAD'],text=True).strip()
            dirty=subprocess.check_output(['git','-C',str(target),'status','--porcelain','--untracked-files=no'],text=True).strip()
            if head!=commit or dirty:raise ValueError('Historical producer checkout is not clean: '+commit)
    command=[sys.executable,'-u','-m','experiments.clip_study','--phase',PHASE,
             '--input-root','/kaggle/input','--out',str(study_root),'--data-dir',str(DATA_ROOT)]
    if PHASE == 'C':command+=['--selection',str(selection)]
    launch_logged(command,cwd=REPO,env={**ENV,'CUDA_VISIBLE_DEVICES':'0,1'},
        log_path=RUN_ROOT/f'clip_study_{PHASE}.log',lane='study',job='clip_'+PHASE,
        lanes=lane_state,paused=session_paused,deadline=VNA_DEADLINE)
finally:
    archive_stop.set();archiver.join(timeout=10);archive_outputs()
report=study_root/'CLIP_STUDY_REPORT.md'
if report.exists():
    from IPython.display import Markdown,display
    display(Markdown(report.read_text()))
if selection.exists():print(selection.read_text())
if session_paused.is_set():print('PAUSED: reattach these study outputs AND all four historical inputs, then rerun.')
if archive_errors:raise RuntimeError(archive_errors)
